# PID Tuning — Interactive Demo

Tune a PID controller against an FOPDT process with live visualization.

**Requirements:** `pip install ipywidgets plotly pandas numpy`

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, FloatSlider, Dropdown

from pypid import PID, FOPDTSimulator, Mode

## Simulation Engine

Runs the PID loop against an FOPDT process with a setpoint step change.

In [ ]:
def simulate(Kc, Ti, Td, K_proc=2.0, tau_proc=60.0, theta_proc=10.0,
             sp=50.0, pv0=50.0, t_final=600.0, dt=1.0,
             sp_step_time=60.0, sp_step_size=10.0):
    """Run PID + FOPDT simulation. Returns a DataFrame."""
    pid = PID(Kc=Kc, Ti=Ti, Td=Td, setpoint=sp,
              output_limits=(0, 100), sample_time=None, time_base='minutes')
    sim = FOPDTSimulator(K=K_proc, tau=tau_proc, theta=theta_proc, y0=pv0)

    # Initialize at steady state
    ss_output = pv0 / K_proc if K_proc != 0 else 50.0
    pid.mode = Mode.MANUAL
    pid.output = ss_output
    pid(pv0, dt=dt)
    pid.mode = Mode.AUTO

    data = []
    pv = pv0
    n_steps = int(t_final / dt)

    for i in range(n_steps):
        t = i * dt
        if t >= sp_step_time:
            pid.setpoint = sp + sp_step_size

        output = pid(pv, dt=dt)
        pv = sim.update(output, dt=dt)
        p, integ, d = pid.components
        data.append({
            'time': t,
            'pv': pv,
            'sp': pid.setpoint,
            'output': output,
            'P': Kc * p,
            'I': Kc * integ,
            'D': Kc * d,
            'error': pid.setpoint - pv,
        })

    return pd.DataFrame(data)

## Plotting Function

In [ ]:
def plot_response(df, title='PID Step Response'):
    """Create a 3-panel interactive plot."""
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        subplot_titles=('Process Variable', 'Controller Output', 'PID Components'),
        vertical_spacing=0.06,
        row_heights=[0.4, 0.3, 0.3],
    )

    # PV and SP
    fig.add_trace(go.Scatter(x=df['time'], y=df['pv'], name='PV',
                             line=dict(color='blue', width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['time'], y=df['sp'], name='SP',
                             line=dict(color='red', width=2, dash='dash')), row=1, col=1)

    # Output
    fig.add_trace(go.Scatter(x=df['time'], y=df['output'], name='Output',
                             line=dict(color='green', width=2)), row=2, col=1)

    # PID Components
    fig.add_trace(go.Scatter(x=df['time'], y=df['P'], name='P term',
                             line=dict(color='orange')), row=3, col=1)
    fig.add_trace(go.Scatter(x=df['time'], y=df['I'], name='I term',
                             line=dict(color='purple')), row=3, col=1)
    fig.add_trace(go.Scatter(x=df['time'], y=df['D'], name='D term',
                             line=dict(color='brown')), row=3, col=1)

    fig.update_layout(height=700, title_text=title, showlegend=True)
    fig.update_xaxes(title_text='Time (seconds)', row=3, col=1)
    fig.update_yaxes(title_text='EU', row=1, col=1)
    fig.update_yaxes(title_text='%', row=2, col=1)
    fig.update_yaxes(title_text='Contribution', row=3, col=1)
    fig.show()

## 2. Interactive Tuning with Sliders

Adjust Kc, Ti, Td and see the response update live.

In [ ]:
@interact(
    Kc=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Kc (gain)'),
    Ti=FloatSlider(value=2.0, min=0.1, max=20.0, step=0.1, description='Ti (min)'),
    Td=FloatSlider(value=0.0, min=0.0, max=5.0, step=0.1, description='Td (min)'),
)
def tune_pid(Kc, Ti, Td):
    df = simulate(Kc=Kc, Ti=Ti, Td=Td)
    plot_response(df, title=f'Kc={Kc:.1f}, Ti={Ti:.1f} min, Td={Td:.1f} min')

## 3. Effect of Gain (Kc) — P-Only Control

Shows offset that exists without integral action.

In [ ]:
fig_p = go.Figure()
for Kc in [0.5, 1.0, 2.0, 4.0]:
    df = simulate(Kc=Kc, Ti=None, Td=None)
    fig_p.add_trace(go.Scatter(x=df['time'], y=df['pv'], name=f'Kc={Kc}'))

fig_p.add_trace(go.Scatter(x=df['time'], y=df['sp'], name='SP',
                           line=dict(color='red', dash='dash')))
fig_p.update_layout(title='P-Only: Higher Kc reduces offset but never eliminates it',
                    xaxis_title='Time (s)', yaxis_title='PV (EU)',
                    height=400)
fig_p.show()

## 4. Adding Integral Action (PI Control)

Integral eliminates offset. Smaller Ti = faster reset = more aggressive.

In [ ]:
fig_pi = go.Figure()
for Ti in [1.0, 2.0, 5.0, 10.0]:
    df = simulate(Kc=1.0, Ti=Ti, Td=None)
    fig_pi.add_trace(go.Scatter(x=df['time'], y=df['pv'], name=f'Ti={Ti} min'))

fig_pi.add_trace(go.Scatter(x=df['time'], y=df['sp'], name='SP',
                            line=dict(color='red', dash='dash')))
fig_pi.update_layout(title='PI Control (Kc=1.0): Effect of Integral Time',
                     xaxis_title='Time (s)', yaxis_title='PV (EU)',
                     height=400)
fig_pi.show()

## 5. Adding Derivative Action (PID Control)

Derivative anticipates change, reduces overshoot on slow processes.

In [ ]:
fig_pid = go.Figure()
for Td in [0.0, 0.3, 0.6, 1.0]:
    label = f'Td={Td} min' if Td > 0 else 'PI only (Td=0)'
    df = simulate(Kc=1.5, Ti=2.0, Td=Td)
    fig_pid.add_trace(go.Scatter(x=df['time'], y=df['pv'], name=label))

fig_pid.add_trace(go.Scatter(x=df['time'], y=df['sp'], name='SP',
                             line=dict(color='red', dash='dash')))
fig_pid.update_layout(title='PID Control (Kc=1.5, Ti=2.0): Effect of Derivative Time',
                      xaxis_title='Time (s)', yaxis_title='PV (EU)',
                      height=400)
fig_pid.show()

## 6. Process Characteristics — What Makes Tuning Hard?

Compare easy (small dead time) vs. hard (large dead time) processes.

In [ ]:
fig_proc = go.Figure()
for theta in [2.0, 10.0, 30.0, 60.0]:
    df = simulate(Kc=1.0, Ti=2.0, Td=0.3, theta_proc=theta)
    fig_proc.add_trace(go.Scatter(x=df['time'], y=df['pv'],
                                  name=f'Dead time={theta}s'))

fig_proc.add_trace(go.Scatter(x=df['time'], y=df['sp'], name='SP',
                              line=dict(color='red', dash='dash')))
fig_proc.update_layout(
    title='Same tuning, different dead times — larger θ = harder to control',
    xaxis_title='Time (s)', yaxis_title='PV (EU)', height=400)
fig_proc.show()

## 7. Full Interactive: Tune Both Controller AND Process

Explore how process parameters affect required tuning.

In [ ]:
@interact(
    Kc=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Kc'),
    Ti=FloatSlider(value=2.0, min=0.1, max=20.0, step=0.1, description='Ti (min)'),
    Td=FloatSlider(value=0.0, min=0.0, max=5.0, step=0.1, description='Td (min)'),
    K_proc=FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description='K (gain)'),
    tau_proc=FloatSlider(value=60.0, min=10.0, max=200.0, step=5.0, description='τ (sec)'),
    theta_proc=FloatSlider(value=10.0, min=0.0, max=60.0, step=1.0, description='θ (sec)'),
)
def full_tuning(Kc, Ti, Td, K_proc, tau_proc, theta_proc):
    df = simulate(Kc=Kc, Ti=Ti, Td=Td,
                  K_proc=K_proc, tau_proc=tau_proc, theta_proc=theta_proc)
    
    # Compute performance metrics
    sp_region = df[df['time'] >= 60.0]
    overshoot = (sp_region['pv'].max() - sp_region['sp'].iloc[0]) / 10.0 * 100
    iae = np.trapz(np.abs(sp_region['error']), sp_region['time'])
    
    title = (f'Kc={Kc:.1f}, Ti={Ti:.1f}min, Td={Td:.1f}min | '
             f'Overshoot={overshoot:.0f}% | IAE={iae:.0f}')
    plot_response(df, title=title)

## 8. Tuning Rules of Thumb

| Rule | Kc | Ti | Td |
|------|----|----|----|
| Start conservative | 0.5/K | 2×τ | 0 |
| Cohen-Coon PI | (τ/K/θ)×(0.9 + θ/12τ) | θ×(30+3θ/τ)/(9+20θ/τ) | 0 |
| Lambda (non-aggressive) | τ/(K×λ) | τ | 0 |

Where λ (lambda) = desired closed-loop time constant, typically 3×θ for robustness.

### Try Lambda Tuning:

In [ ]:
# Process parameters
K, tau, theta = 2.0, 60.0, 10.0

# Lambda tuning: lambda = 3 * theta
lam = 3 * theta
Kc_lambda = tau / (K * lam)
Ti_lambda = tau / 60.0  # convert seconds to minutes

print(f'Process: K={K}, τ={tau}s, θ={theta}s')
print(f'Lambda tuning (λ={lam}s): Kc={Kc_lambda:.2f}, Ti={Ti_lambda:.2f} min')

df = simulate(Kc=Kc_lambda, Ti=Ti_lambda, Td=None,
              K_proc=K, tau_proc=tau, theta_proc=theta)
plot_response(df, title=f'Lambda Tuning: Kc={Kc_lambda:.2f}, Ti={Ti_lambda:.2f} min')